# 01 — Data ingestion and profiling

## Goal
Create a reproducible local copy of the real Olist data, load **the files that are actually present**, and establish a factual quality baseline. This notebook intentionally makes no cleaning changes.

**Business context:** Olist is a Brazilian marketplace. The raw source has multiple related tables, so we must validate each table's grain and keys before any merge or revenue calculation.

**Rule:** every finding in later notebooks must be traceable to this raw-data baseline.

In [2]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
sns.set_theme(style='whitegrid', palette='deep')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for folder in (RAW_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f'Python version: {sys.version.split()[0]}')
print(f'Project root: {PROJECT_ROOT}')

Python version: 3.12.10
Project root: C:\Users\Muhamed\Documents\Codex\2026-08-11\referenced-chatgpt-conversation-this-is-an


## 1. Confirm the supplied raw data

The project uses the six CSV files supplied in `data/raw/`. We confirm their presence before reading anything. The source dataset is the Brazilian Olist e-commerce dataset; our subsequent checks use the delivered files and their actual schema.

In [3]:
expected_files = {'customers.csv', 'orders.csv', 'order_items.csv', 'order_payments.csv', 'products.csv', 'seller.csv'}
found_files = {file.name for file in RAW_DIR.glob('*.csv')}
missing_files = expected_files - found_files
if missing_files:
    raise FileNotFoundError(f'Missing raw files: {sorted(missing_files)}. Found: {sorted(found_files)}')

print(f'All {len(expected_files)} supplied raw files are present in: {RAW_DIR}')

All 6 supplied raw files are present in: C:\Users\Muhamed\Documents\Codex\2026-08-11\referenced-chatgpt-conversation-this-is-an\data\raw


## 2. Create a raw-file inventory

Why this matters: file names, columns, row counts, encoding, and delimiter issues are properties of the delivered extract—not assumptions. The inventory is our first reproducible audit artifact.

In [4]:
raw_files = sorted(RAW_DIR.glob('*.csv'))
if not raw_files:
    raise FileNotFoundError('No raw CSVs found. Run Step 1 successfully before continuing.')

inventory = pd.DataFrame([{
    'file_name': file.name,
    'size_mb': round(file.stat().st_size / 1024**2, 2),
    'rows': sum(1 for _ in file.open(encoding='utf-8')) - 1
} for file in raw_files]).sort_values('file_name')

inventory.to_csv(TABLE_DIR / 'raw_file_inventory.csv', index=False)
display(inventory)

,file_name,size_mb,rows
0,customers.csv,8.62,99441
1,order_items.csv,14.72,112650
2,order_payments.csv,5.51,103886
3,orders.csv,16.84,99441
4,products.csv,2.27,32951
5,seller.csv,0.17,3095


## 3. Load CSVs dynamically and inspect their actual schema

The dictionary keys come from the downloaded filenames. This prevents us from inventing a schema. IDs remain strings to preserve leading zeros and to avoid accidental numeric conversion.

In [5]:
tables = {file.stem.replace('_dataset', ''): pd.read_csv(file, dtype=str) for file in raw_files}

schema_rows = []
for table_name, frame in tables.items():
    for column_name in frame.columns:
        schema_rows.append({
            'table': table_name,
            'column': column_name,
            'raw_dtype': str(frame[column_name].dtype),
            'missing_rows': int(frame[column_name].isna().sum()),
            'missing_pct': round(frame[column_name].isna().mean() * 100, 2),
            'distinct_values': int(frame[column_name].nunique(dropna=True))
        })

schema_report = pd.DataFrame(schema_rows).sort_values(['table', 'column'])
schema_report.to_csv(TABLE_DIR / 'raw_schema_profile.csv', index=False)
display(schema_report)

for table_name, frame in tables.items():
    print(f'\n{table_name}: {frame.shape[0]:,} rows × {frame.shape[1]} columns')
    display(frame.head(3))

,table,column,raw_dtype,missing_rows,missing_pct,distinct_values
3,customers,customer_city,object,0,0.00,4119
0,customers,customer_id,object,0,0.00,99441
4,customers,customer_state,object,0,0.00,27
1,customers,customer_unique_id,object,0,0.00,96096
2,customers,customer_zip_code_prefix,object,0,0.00,14994
11,order_items,freight_value,object,0,0.00,6999
5,order_items,order_id,object,0,0.00,98666
6,order_items,order_item_id,object,0,0.00,21
10,order_items,price,object,0,0.00,5968
7,order_items,product_id,object,0,0.00,32951



customers: 99,441 rows × 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP



order_items: 112,650 rows × 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87


...

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



products: 32,951 rows × 9 columns


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15



seller: 3,095 rows × 4 columns


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


## 4. Establish the expected relational model—only after verifying columns

This is a contract for the supplied Olist extract. First, verify that each referenced file/column exists. Next, we will test uniqueness and orphan records. A failed check is a finding to investigate, not a reason to silently delete rows.

In [6]:
# Map canonical source filename stems to short analytical names after inspecting `tables.keys()`.
required_tables = {
    'customers': ['customer_id', 'customer_unique_id'],
    'orders': ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at'],
    'order_items': ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value'],
    'order_payments': ['order_id', 'payment_sequential', 'payment_type', 'payment_value'],
    'products': ['product_id', 'product_category_name'],
    'seller': ['seller_id']
}

available = set(tables)
missing_tables = set(required_tables) - available
if missing_tables:
    raise KeyError(f'Expected Olist tables are missing: {missing_tables}. Available: {sorted(available)}')

missing_columns = {
    table: sorted(set(columns) - set(tables[table].columns))
    for table, columns in required_tables.items()
    if set(columns) - set(tables[table].columns)
}
if missing_columns:
    raise KeyError(f'Unexpected source schema: {missing_columns}')

print('Schema contract passed. Continue to key and relationship checks.')

Schema contract passed. Continue to key and relationship checks.


In [7]:
# Grain checks: exact keys are based on the real Olist documentation and validated against the loaded data.
key_definitions = {
    'customers': ['customer_id'],
    'orders': ['order_id'],
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'products': ['product_id'],
    'seller': ['seller_id']
}

key_report = pd.DataFrame([{
    'table': table,
    'candidate_key': ' + '.join(key),
    'rows': len(tables[table]),
    'duplicate_key_rows': int(tables[table].duplicated(key).sum()),
    'null_key_rows': int(tables[table][key].isna().any(axis=1).sum())
} for table, key in key_definitions.items()])
key_report.to_csv(TABLE_DIR / 'key_quality_report.csv', index=False)
display(key_report)

,table,candidate_key,rows,duplicate_key_rows,null_key_rows
0,customers,customer_id,99441,0,0
1,orders,order_id,99441,0,0
2,order_items,order_id + order_item_id,112650,0,0
3,order_payments,order_id + payment_sequential,103886,0,0
4,products,product_id,32951,0,0
5,seller,seller_id,3095,0,0


In [8]:
# Foreign-key checks. `orphan_rows` must be reviewed before any cleaning decision.
relationships = [
    ('orders', 'customer_id', 'customers', 'customer_id'),
    ('order_items', 'order_id', 'orders', 'order_id'),
    ('order_items', 'product_id', 'products', 'product_id'),
    ('order_items', 'seller_id', 'seller', 'seller_id'),
    ('order_payments', 'order_id', 'orders', 'order_id')
]

relationship_report = []
for child_table, child_key, parent_table, parent_key in relationships:
    child_values = tables[child_table][child_key].dropna()
    parent_values = set(tables[parent_table][parent_key].dropna())
    orphan_count = int((~child_values.isin(parent_values)).sum())
    relationship_report.append({
        'relationship': f'{child_table}.{child_key} → {parent_table}.{parent_key}',
        'child_non_null_rows': len(child_values),
        'orphan_rows': orphan_count,
        'orphan_pct': round(orphan_count / len(child_values) * 100, 4) if len(child_values) else np.nan
    })

relationship_report = pd.DataFrame(relationship_report)
relationship_report.to_csv(TABLE_DIR / 'relationship_quality_report.csv', index=False)
display(relationship_report)

,relationship,child_non_null_rows,orphan_rows,orphan_pct
0,orders.customer_id → customers.customer_id,99441,0,0.00
1,order_items.order_id → orders.order_id,112650,0,0.00
2,order_items.product_id → products.product_id,112650,0,0.00
3,order_items.seller_id → seller.seller_id,112650,0,0.00
4,order_payments.order_id → orders.order_id,103886,0,0.00
